# RM2 Sentiment Analysis for Goals (LEGACY_SENTIMENT_V1)

Notebook ini adalah entrypoint legacy untuk RM2 Goals V1. Hasil final Sentimen V2 berada di `output/rm2_sentiment/final_v2/` dan script kanonisnya berada di `scripts/apply_rm2_sentiment_v2_final_inference.py`.


## Prinsip Metodologis

- RM1, LCN, HCC membership, Louvain, FSA_V, brand labeling RM1, dan Co-Similarity RM1 hanya dibaca sebagai input terlindungi.
- Final prediction tidak boleh berasal dari rule-based fallback (`ALLOW_RULE_BASED_FINAL = False`).
- Pipeline membandingkan `minimal_raw` dan `social_normalized`, dua transformer, transformer-only ensemble, serta baseline klasik TF-IDF + LinearSVC terkalibrasi sebagai eksperimen pseudo-label yang tidak eligible sebagai final tanpa label manusia.
- Sebelum label manusia tersedia, evaluasi domain memakai **heuristic pseudo-label / deterministic lexical reference** sebagai diagnostic internal; `manual_label` dan label manusia tetap kosong untuk anotasi manusia mendatang.
- Goal orientation adalah orientasi pesan berbasis distribusi sentimen komentar yang teramati, bukan bukti niat psikologis, hubungan komersial, pembayaran, atau koordinasi terencana.


In [1]:
from pathlib import Path
import json
import sys


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "dataset.csv").exists() and (candidate / "scripts").exists():
            return candidate
    raise RuntimeError("Project root tidak ditemukan. Jalankan notebook dari dalam repository Analisis-Astroturfing.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.project_paths import PROJECT_ROOT as CANONICAL_PROJECT_ROOT
from scripts.rm2_sentiment_goals_pipeline import run_pipeline

ROOT = CANONICAL_PROJECT_ROOT
RUN_MODE = "legacy"
print("LEGACY_SENTIMENT_V1 notebook. Final Sentimen V2 berada di output/rm2_sentiment/final_v2/.")


In [2]:
result = run_pipeline(ROOT)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

C:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Python312\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Python312\Lib\site-packages\sklearn\metrics\_c

RM2 SENTIMENT GOALS PIPELINE COMPLETE
- selected pipeline: ensemble_human_A0.00_B0.25_C0.75 (social_normalized)
- confidence threshold: 0.430
- comment rows: 33,847
- HCC comments: 1,009
- HCC count: 42
- overall pipeline status: VALIDATED
- human validation completed: True
- development macro-F1: 0.7435
- locked-test human_adjudicated_label macro-F1: 0.5830 (95% CI 0.5056-0.6500)
- sentiment status counts:
sentiment_status
Evaluable    28989
Uncertain     3138
No Text       1720
- HCC goal counts:
goal_orientation
Mixed Goals                 18
Promotional / Supportive    11
Neutral Engagement          10
Polarized / Contested        2
Critical / Complaint         1
- HCC goal confidence counts:
goal_confidence
Low       19
High      12
Medium    11
- deterministic rule reproducibility (not annotation reliability):
 sample_set  n_comments  deterministic_rule_reproducibility  cohen_kappa_deterministic_reproducibility  disagreement_count  uncertainty_rate  no_text_rate  passes_independe

In [3]:
print(json.dumps(result, indent=2, ensure_ascii=False))


{
  "selected_candidate": "ensemble_human_A0.00_B0.25_C0.75",
  "selected_variant": "social_normalized",
  "selected_threshold": 0.43,
  "overall_pipeline_status": "VALIDATED",
  "human_validation_completed": true,
  "development_macro_f1": 0.7435448942195569,
  "locked_test_macro_f1": 0.5829815793573748,
  "locked_test_macro_f1_ci": [
    0.5056449807072054,
    0.6500293650058356
  ],
  "locked_test_neutral_recall": 0.6934306569343066,
  "comment_rows": 33847,
  "hcc_comments": 1009,
  "hcc_goal_counts": {
    "Mixed Goals": 18,
    "Promotional / Supportive": 11,
    "Neutral Engagement": 10,
    "Polarized / Contested": 2,
    "Critical / Complaint": 1
  },
  "goal_confidence_counts": {
    "Low": 19,
    "High": 12,
    "Medium": 11
  },
  "low_confidence_hcc": [
    "1",
    "2",
    "5",
    "17",
    "18",
    "34",
    "45",
    "46",
    "48",
    "60",
    "63",
    "74",
    "75",
    "103",
    "117",
    "134",
    "150",
    "182",
    "183"
  ],
  "rm1_unchanged": true


## Output Utama

Tabel utama disimpan di `output/rm2_sentiment/tables/`, visualisasi inti di `output/rm2_sentiment/visualisasi/`, dan file Gephi sentimen HCC di `output/rm2_sentiment/gephi/`.

Visualisasi utama dibatasi pada tiga file: `sentiment_validation_confusion_matrix.png`, `sentiment_hcc_vs_nonhcc_100pct.png`, dan `hcc_goal_orientation_confidence.png`. WordCloud tidak digunakan sebagai validasi sentimen.


## Batas Interpretasi

- Sentimen hanya indikator orientasi pesan dan kecenderungan narasi.
- `goal_orientation` tidak membuktikan niat, afiliasi, pembayaran, instruksi, hubungan komersial, pengaruh kausal, buzzer, atau astroturfing.
- Human validation 600 komentar sudah lengkap dan digunakan sebagai reference label untuk evaluasi comment-level sentiment model.
- Diagnostic agreement goal pada level HCC tetap dilaporkan terpisah; confidence/stability bukan akurasi atau bukti kebenaran label goal.
- HCC sudah final dari RM1; sentimen tidak digunakan untuk membentuk HCC.
